# LASCO Data Download, Reduction, And Visualization

This notebook is the interactive real-data workflow. Fill in the time range, detector, output folders, and calibration directory. Then use the buttons to search, download, reduce, and visualize.

The search/download step uses `sunpy.net.Fido`, so public archive availability can vary. You can also skip search/download and use the local-file pattern box to reduce FITS files that are already on disk.

In [ ]:
from glob import glob
from pathlib import Path

import ipywidgets as widgets
from IPython.display import Image, display

from sohopy.lasco import LASCOConfig, plot_fits_image, reduce_level_1

## 1. Select Data

In [ ]:
start_time = widgets.Text(value="2005-01-01 00:00", description="Start UTC", layout=widgets.Layout(width="320px"))
end_time = widgets.Text(value="2005-01-01 02:00", description="End UTC", layout=widgets.Layout(width="320px"))
detector = widgets.Dropdown(options=["C2", "C3"], value="C2", description="Detector")
detector_in_search = widgets.Checkbox(value=True, description="Use detector in public search")
filter_name = widgets.Text(value="", description="Filter", placeholder="optional FITS header filter", layout=widgets.Layout(width="320px"))
polar_name = widgets.Text(value="", description="Polar", placeholder="optional FITS header polarizer", layout=widgets.Layout(width="320px"))
max_files = widgets.IntSlider(value=3, min=1, max=25, step=1, description="Max files")

download_dir = widgets.Text(value="data/lasco/downloads", description="Download dir", layout=widgets.Layout(width="720px"))
calibration_root = widgets.Text(value="data/lasco/calib", description="Calib dir", layout=widgets.Layout(width="720px"))
level1_dir = widgets.Text(value="data/lasco/level1", description="Level 1 dir", layout=widgets.Layout(width="720px"))
preview_dir = widgets.Text(value="data/lasco/previews", description="Preview dir", layout=widgets.Layout(width="720px"))
local_pattern = widgets.Text(value="", description="Local FITS", placeholder="optional glob, e.g. data/lasco/downloads/*.fts", layout=widgets.Layout(width="720px"))

search_button = widgets.Button(description="Search public data", button_style="info")
download_button = widgets.Button(description="Download selected", button_style="success")
reduce_button = widgets.Button(description="Reduce local FITS", button_style="warning")
visualize_button = widgets.Button(description="Visualize Level 1", button_style="primary")
workflow_output = widgets.Output()

display(widgets.VBox([
    widgets.HBox([start_time, end_time]),
    widgets.HBox([detector, max_files]),
    detector_in_search,
    widgets.HBox([filter_name, polar_name]),
    download_dir,
    calibration_root,
    level1_dir,
    preview_dir,
    local_pattern,
    widgets.HBox([search_button, download_button, reduce_button, visualize_button]),
    workflow_output,
]))

search_response = None
downloaded_files = []
level1_files = []

## 2. Search And Download

The search uses broad LASCO constraints first. Filter/polarizer matching is usually more reliable after download because archive metadata are not always uniform across services.

In [ ]:
def _sunpy_query():
    from sunpy.net import Fido, attrs as a

    query = [a.Time(start_time.value, end_time.value), a.Instrument("LASCO")]
    # Detector is best-effort because providers differ in metadata support.
    if detector_in_search.value and hasattr(a, "Detector"):
        query.append(a.Detector(detector.value))
    return Fido, query


def search_clicked(_):
    global search_response
    workflow_output.clear_output()
    with workflow_output:
        print(f"Searching LASCO {detector.value} from {start_time.value} to {end_time.value}...")
        Fido, query = _sunpy_query()
        search_response = Fido.search(*query)
        print(search_response)
        if len(search_response) == 0:
            print("No results. Try a wider time range or uncheck detector constraints.")


def download_clicked(_):
    global downloaded_files
    workflow_output.clear_output()
    with workflow_output:
        if search_response is None or len(search_response) == 0:
            print("No search results yet. Press 'Search public data' first.")
            return
        out_dir = Path(download_dir.value).expanduser()
        out_dir.mkdir(parents=True, exist_ok=True)
        Fido, _ = _sunpy_query()
        first_provider = search_response[0]
        selected = first_provider[: int(max_files.value)]
        print(f"Downloading up to {len(selected)} files into {out_dir}")
        paths = Fido.fetch(selected, path=str(out_dir / "{file}"))
        downloaded_files = [Path(path) for path in paths]
        for path in downloaded_files:
            print(path)


search_button.on_click(search_clicked)
download_button.on_click(download_clicked)

## 3. Reduce Local FITS Files

If you already have files, fill in **Local FITS** with a glob pattern. If it is blank, the notebook uses the files downloaded in the previous step.

In [ ]:
def _candidate_inputs():
    if local_pattern.value.strip():
        return [Path(path) for path in sorted(glob(local_pattern.value.strip()))]
    return list(downloaded_files)


def _header_matches(path):
    from astropy.io import fits

    with fits.open(path, memmap=False) as hdul:
        header = hdul[0].header
    if str(header.get("DETECTOR", "")).strip().upper() not in {"", detector.value.upper()}:
        return False
    if filter_name.value.strip() and str(header.get("FILTER", "")).strip().upper() != filter_name.value.strip().upper():
        return False
    if polar_name.value.strip() and str(header.get("POLAR", "")).strip().upper() != polar_name.value.strip().upper():
        return False
    return True


def reduce_clicked(_):
    global level1_files
    workflow_output.clear_output()
    with workflow_output:
        inputs = [path for path in _candidate_inputs() if _header_matches(path)]
        if not inputs:
            print("No local FITS inputs matched. Download files or set the Local FITS glob.")
            return
        out_dir = Path(level1_dir.value).expanduser()
        out_dir.mkdir(parents=True, exist_ok=True)
        config = LASCOConfig(calibration_root=Path(calibration_root.value).expanduser())
        level1_files = []
        for path in inputs:
            output = out_dir / f"{path.stem}_l1.fts"
            print(f"Reducing {path} -> {output}")
            try:
                reduce_level_1(path, config=config, output_path=output, overwrite=True)
                level1_files.append(output)
            except Exception as exc:
                print(f"  failed: {exc}")
        print(f"Reduced {len(level1_files)} files.")


reduce_button.on_click(reduce_clicked)

## 4. Visualize

In [ ]:
def visualize_clicked(_):
    workflow_output.clear_output()
    with workflow_output:
        files = level1_files
        if not files:
            files = [Path(path) for path in sorted(glob(str(Path(level1_dir.value).expanduser() / "*.fts")))]
        if not files:
            print("No Level 1 files found. Run reduction first.")
            return
        out_dir = Path(preview_dir.value).expanduser()
        out_dir.mkdir(parents=True, exist_ok=True)
        for path in files[: int(max_files.value)]:
            png = out_dir / f"{path.stem}.png"
            plot_fits_image(path, png, title=path.name)
            print(png)
            display(Image(filename=str(png)))


visualize_button.on_click(visualize_clicked)

## Tips

- Start with a short time range and a small max-file count.
- Make sure the calibration cache is prepared before reducing.
- If archive metadata are sparse, download first and use the filter/polarizer boxes as local FITS-header filters.
- For guaranteed no-network behavior, use `00_synthetic_full_workflow.ipynb`.